# VitaNexus-RX — Resume HGNN from epoch 11 and finish
This recovery notebook restores the verified Drive backup when necessary, confirms epochs 1–10 are durable, resumes at epoch 11, completes selection/final-refit/calibration/2026 holdout evaluation, promotes the HGNN artifacts, and exports a combined LightGBM+HGNN inference bundle. It never retrains LightGBM.

Before **Run all**, select **Runtime → Change runtime type → Runtime version 2026.07 (Python 3.12) → T4 GPU** (or L4/A100). Upload the original backup ZIP to **My Drive** when using a new Google account.

## 1. Fixed recovery configuration — do not change

In [ ]:
import sys
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Runtime -> Change runtime type -> Runtime Version 2026.07 (Python 3.12), choose a GPU, reconnect, then Run all.')
DRIVE_ROOT = '/content/drive/MyDrive/VitaNexus-RX-ML'
BACKUP_ZIP_NAME = 'VitaNexus-RX-ML-20260904T151940Z-1-001.zip'
BACKUP_SHA256 = '8d77f2637712fd4c78ad4b35659dd127109c525f3e4e4bea614b7dde9400a631'
EXPECTED_HGNN_RUN = '0c32c0c81bc415a1dbe2'
EXPECTED_COMPLETED_EPOCHS = 10
REPOSITORY_URL = 'https://github.com/Sravanramaraju/VitaNexus-RX.git'
BRANCH = 'codex/faers-ml-clinical-integration'
MINIMUM_RESUME_FIX_COMMIT = '16fccc652488f4ea31ad2fe3d1732b839c1f164f'
REPOSITORY = '/content/VitaNexus-RX'
LOCAL_WORK = '/content/vitanexus-ml-work'

## 2. Mount Google Drive — authorize the account containing the existing folder or uploaded ZIP

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Acquire the committed repository branch containing the checkpoint repair

In [ ]:
import pathlib, subprocess
repo = pathlib.Path(REPOSITORY)
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, REPOSITORY], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=repo, check=True)
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo, text=True).strip():
        raise RuntimeError('Existing Colab checkout is dirty; use a fresh runtime instead of discarding files.')
    subprocess.run(['git', 'checkout', BRANCH], cwd=repo, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=repo, check=True)
subprocess.run(['git', 'merge-base', '--is-ancestor', MINIMUM_RESUME_FIX_COMMIT, 'HEAD'], cwd=repo, check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip())

## 4. Restore the verified old Drive snapshot when using a new account

In [ ]:
from pathlib import Path
import subprocess, sys
my_drive = Path('/content/drive/MyDrive')
backup = my_drive / BACKUP_ZIP_NAME
drive_root = Path(DRIVE_ROOT)
existing_checkpoint = drive_root / 'training_state/training_runs/hgnn' / EXPECTED_HGNN_RUN / 'hgnn_selection_latest.pt'
if backup.is_file():
    subprocess.run([
        sys.executable, str(repo / 'ml/scripts/colab_training_snapshot.py'), 'restore-drive-download',
        '--bundle', str(backup), '--my-drive', str(my_drive), '--expected-sha256', BACKUP_SHA256,
    ], check=True)
elif existing_checkpoint.is_file():
    print('Existing persistent Drive state found; backup extraction is unnecessary.')
else:
    matches = sorted(my_drive.glob('VitaNexus-RX-ML-*.zip'))
    raise FileNotFoundError(f'Upload {BACKUP_ZIP_NAME} to My Drive. Expected {backup}; other candidates={matches}')

## 5. Install constrained dependencies without replacing Colab's CUDA PyTorch

In [ ]:
import importlib.metadata, os, signal, subprocess, sys, time
from pathlib import Path
requirements = Path(REPOSITORY) / 'ml' / 'requirements-colab.txt'
pins = dict(line.split('==', 1) for line in requirements.read_text().splitlines() if '==' in line and not line.lstrip().startswith('#'))
installed = {}
for package in pins:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed[package] = None
changes = {package: {'installed': installed[package], 'required': required} for package, required in pins.items() if installed[package] != required}
probe_code = 'import numpy; from numpy._core.umath import _slice; import pandas, pyarrow, scipy, sklearn, lightgbm'
probe = subprocess.run([sys.executable, '-c', probe_code], capture_output=True, text=True)
broken_stack = probe.returncode != 0
if changes or broken_stack:
    if changes:
        print(f'Installing constrained dependencies: {changes}', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
    if broken_stack:
        print(f'Scientific-stack import probe failed; repairing binary packages:\n{probe.stderr}', flush=True)
        repair = [f'{name}=={pins[name]}' for name in ('numpy', 'pandas', 'pyarrow', 'scipy', 'scikit-learn', 'lightgbm')]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps', *repair], check=True)
    print('DEPENDENCIES READY. Colab is restarting once to prevent mixed binary modules. After reconnection, choose Runtime -> Run all again.', flush=True)
    time.sleep(2)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Constrained dependencies and binary import probe passed; no restart required.')

## 6. Resolve paths and stage the verified immutable FAERS data

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, f'{REPOSITORY}/ml/colab')
from colab_common import ColabPaths, configure, stage_dataset
paths = ColabPaths(Path(DRIVE_ROOT), Path(REPOSITORY), Path(LOCAL_WORK))
configure(paths)
stage_dataset(paths)

## 7. Strict resume preflight — require CUDA and a contiguous 10–20 epoch checkpoint

In [ ]:
import json, torch
from colab_common import hgnn_preflight
preflight = hgnn_preflight(paths, BRANCH)
if not preflight['runtime']['cudaAvailable']:
    raise RuntimeError('CUDA is unavailable. Stop, select a T4/L4/A100 GPU runtime, reconnect, and Run all.')
from vitanexus_ml.cli import hgnn_training_status
status = hgnn_training_status()
assert status['run'] == EXPECTED_HGNN_RUN, status
completed = status['selection']['completedEpochs']
total = status['selection']['totalEpochs']
assert EXPECTED_COMPLETED_EPOCHS <= completed <= 20, status
assert total == 20, status
if completed < total:
    checkpoint_path = Path(status['statePath']).parent / 'hgnn_selection_latest.pt'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    history_epochs = [item['epoch'] for item in checkpoint.get('history', [])]
    assert history_epochs == list(range(1, completed + 1)), history_epochs
    assert int(checkpoint['nextEpoch']) == completed, checkpoint['nextEpoch']
    print(f'HGNN RESUME VERIFIED: {completed}/20 epochs safe; next displayed epoch={completed + 1}/20')
else:
    print('HGNN selection is already 20/20; remaining final stages will resume.')
print('CUDA ACTIVE:', preflight['runtime']['gpu'], preflight['runtime']['gpuMemoryGB'], 'GB')

## 8. Resume selection, then final-refit, calibrate and evaluate the full HGNN

In [ ]:
import os, subprocess, sys
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    [sys.executable, '-u', '-m', 'vitanexus_ml.cli', 'train-hgnn'],
    cwd=REPOSITORY, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'HGNN training exited with status {return_code}; the complete traceback is printed above. Rerun from section 7 after correcting the reported cause.')

## 9. Verify full HGNN promotion and untouched 2026 holdout evaluation

In [ ]:
import json
from vitanexus_ml.cli import hgnn_training_status
final_status = hgnn_training_status()
assert final_status['status'] == 'COMPLETE', final_status
manifest = json.loads((paths.drive_models / 'hgnn_training_manifest.json').read_text())
assert manifest['fastMode'] is False and manifest['fullFinalData'] is True
assert manifest['selectionTrainWindow'] == ['2022Q1', '2024Q4']
assert manifest['validationWindow'] == ['2025Q1', '2025Q2']
assert manifest['finalFitWindow'] == ['2022Q1', '2025Q2']
assert manifest['calibrationWindow'] == ['2025Q3', '2025Q3']
assert manifest['holdoutWindow'] == ['2026Q1', '2026Q2']
metrics = json.loads((paths.drive_reports / 'hgnn_metrics.json').read_text())
assert metrics['fastMode'] is False and metrics['fullFinalData'] is True
assert 'holdout2026' in metrics and len(metrics['perLabelHoldout2026']) == 100
print('HGNN FULL TRAINING COMPLETE AND PROMOTED:', json.dumps(metrics['holdout2026'], indent=2))

## 10. Export and ZIP the combined verified LightGBM + HGNN inference bundle

In [ ]:
import shutil, zipfile
from vitanexus_ml.artifact_bundle import export_inference_bundle, verify_inference_bundle
output = paths.drive_exports / 'vitanexus_full_inference'
if (output / 'inference_bundle_manifest.json').exists():
    result = verify_inference_bundle(output, require_all=True)
else:
    result = export_inference_bundle(paths.drive_models, paths.drive_reports, output, component='all')
verified = verify_inference_bundle(output, require_all=True)
archive_base = paths.drive_exports / 'vitanexus_full_inference'
archive_path = Path(shutil.make_archive(
    str(paths.drive_exports / 'vitanexus_full_inference_bundle'), 'zip',
    root_dir=paths.drive_exports, base_dir=output.name,
))
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
print(json.dumps(verified, indent=2))
print('COMBINED BUNDLE ZIP READY:', archive_path, 'bytes=', archive_path.stat().st_size)

## 11. Completion condition
The pipeline is complete only when section 9 prints **HGNN FULL TRAINING COMPLETE AND PROMOTED** and section 10 prints **COMBINED BUNDLE ZIP READY**. Download `VitaNexus-RX-ML/exports/vitanexus_full_inference_bundle.zip`; it is the combined production candidate that will be verified again before local runtime integration. If Colab disconnects, reconnect and **Run all**: completed Drive stages and epoch checkpoints are reused.